# Sheets → BananaDrum

Convert samba percursion notes written in Google Sheets into a [BananaDrum](https://www.bannadrum.net) link.

---

Fill in the form below and click ▶ to generate your BananaDrum link.  
The first time you run it, the tool takes about 30 seconds to set itself up.

In [1]:
#@title Sheets → BananaDrum { display-mode: "form", run: "auto" }
#@markdown Open the Google Sheets file, navigate to the worksheet that contains the samba break notes and copy the link from the address bar.
#@markdown The Google Sheets file must be set to *Anyone with the link can view*.
#@markdown
#@markdown **Step 1) (mandatory)** — Paste the copied link below in the field sheets_link.
#@markdown
#@markdown **Step 2) (optional)** — If the worksheet contains more than one break, choose which one to convert (1 = first break in the sheet, 2 = second, etc.).
#@markdown
#@markdown **Step 3) (optional)** — You may choose the playback speed of the Bananadrum link in BPM . (default = 110).
#@markdown
#@markdown ---

sheets_link = "https://docs.google.com/spreadsheets/d/18-Spm7SOXxE5TI8wAOpZGkZMlnkCEZAaio9LkiP_fhc/edit?gid=2085889999#gid=2085889999" #@param {type:"string", placeholder:"Paste the Google Sheets link here"}
break_number = 1 #@param {type:"integer", placeholder:"Enter the number of the break you want to convert (e.g. 1)"}
tempo = "110" #@param {type:"string", placeholder:"e.g. 110"}

import importlib
if importlib.util.find_spec("sheets_to_banana") is None:
    print("\u231b Preparing  the conversion tool — this only happens once and takes about 30 seconds...")
    import subprocess, shutil
    if shutil.which("uv") is None:
        subprocess.run(["pip", "install", "uv", "-q"], check=True, capture_output=True)
    subprocess.run(
        ["uv", "pip", "install",
         "git+https://github.com/bruno-ritmista/samba@c8eaa29da242271851978d553fe6a5bdf7ae2e1b#subdirectory=sheets_to_banana",
         "--system", "-q"],
        check=True, capture_output=True
    )
    print("\u2705 Conversion tool ready.")

import threading, time
if not any(t.name == "colab_keepalive" for t in threading.enumerate()):
    def _keep_alive():
        while True:
            time.sleep(60)
    _t = threading.Thread(target=_keep_alive, name="colab_keepalive", daemon=True)
    _t.start()

import logging as _logging
import re

def _setup_notebook_logging():
    class _FriendlyFormatter(_logging.Formatter):
        def format(self, record):
            if record.levelno == _logging.WARNING:
                return f"\u26a0\ufe0f  {record.getMessage()}"
            return record.getMessage()

    _h = _logging.StreamHandler()
    _h.setFormatter(_FriendlyFormatter())
    _pkg = _logging.getLogger("sheets_to_banana")
    _pkg.handlers.clear()
    _pkg.addHandler(_h)
    _pkg.propagate = False
    _pkg.setLevel(_logging.WARNING)

_setup_notebook_logging()

def _clean_break_name(name):
    cleaned = re.sub(r'\s*\(.*\)\s*$', '', name).strip()
    cleaned = re.sub(r'(?i)\s*[-\s_]*banana[-\s_]*drum[-\s_]*', '', cleaned).strip(' \t-_')
    return cleaned

def _run():
    val = sheets_link.strip()

    if not val:
        print("Paste a Google Sheets link in the field above, then run this cell.")
        return

    if re.search(r'/spreadsheets/d/([a-zA-Z0-9_-]+)', val):
        full_url = val
    elif re.fullmatch(r'[a-zA-Z0-9_-]+', val):
        full_url = f"https://docs.google.com/spreadsheets/d/{val}"
    else:
        print("\u274c That doesn't look like a Google Sheets link. "
              "Please copy the link from the sheet and try again.")
        return

    tempo_clean = re.sub(r'(?i)\s*bpm\s*', '', str(tempo)).strip()
    try:
        tempo_int = int(tempo_clean)
    except ValueError:
        print("\u274c Tempo must be a number (e.g. 120 or \"120 bpm\"). Please fix the Tempo field and try again.")
        return

    try:
        from sheets_to_banana.fetch import fetch_csv
        from sheets_to_banana.parse import parse_sheet
        from sheets_to_banana.mapping import map_break
        from sheets_to_banana.encode import encode_url
    except ImportError:
        print("\u274c The conversion tool failed to load. Please reload the page and try again.")
        return

    try:
        csv_text = fetch_csv(full_url)
    except Exception:
        print("\u274c Could not read your Google Sheet. "
              "Please check the link and try again.")
        return

    breaks = parse_sheet(csv_text)
    if not breaks:
        print("\u274c No breaks found in the sheet. "
              "Please check that the sheet has the expected format.")
        return

    if break_number < 1 or break_number > len(breaks):
        print(f"\u274c Break {break_number} not found. "
              f"This sheet has {len(breaks)} break(s): 1\u2013{len(breaks)}.")
        return

    brk = breaks[break_number - 1]
    tracks = map_break(brk)
    if not tracks:
        print(f"\u274c Break {break_number} \"{brk.name}\" has no recognised instruments.")
        return

    n_bars = max(len(t.notes) for t in tracks) // 16
    clean_name = _clean_break_name(brk.name)
    url = encode_url(tracks, tempo=tempo_int, n_bars=n_bars)

    print(f"\u2705 BananaDrum link created successfully for \"{clean_name}\" "
          f"including {n_bars} bars and {len(tracks)} instruments. "
          f"Click the link below to open:")
    print()
    print("\U0001f517 " + url)

_run()

⚠️  Note '.' for instrument 'Repique de bossa' is not supported by the conversion tool. It will be skipped.
⚠️  Note '.' for instrument 'Repique de bossa' is not supported by the conversion tool. It will be skipped.
⚠️  Note 'K' for instrument 'Caixa' is not supported by the conversion tool. It will be skipped.
⚠️  Note 'xo' for instrument 'Repique de bossa' is not supported by the conversion tool. It will be skipped.
⚠️  Note 'o' for instrument 'Repique de bossa' is not supported by the conversion tool. It will be skipped.
⚠️  Instrument 'Repique de bossa' has a note ('/ o / o / o') that does not seem to be in 4/4 nor 6/8 time signature, so it is not supported by the conversion tool yet. It will be skipped.


✅ BananaDrum link created successfully for "Break "Caranguejo"" including 16 bars and 9 instruments. Click the link below to open:

🔗 https://bananadrum.net/?a2=4-4.110.16.1-4.16.98pKptmhW2ZZXNsjAI19SA3E7r~qEgZGWtV4HbJUkwGHi0PJPXHzofJPBw_Opzox3M1-9h9L.81vJSCjiP_4MZuhag2_rD5OfVvfzJEBP4IEPC_VXoaoqnFZLci5S70YnlddFXNKfJ5DML-9h9L.78pJQ6WXqswTZppWhemPB5RP3Tkz2kqEnzVHUTpvn4aUh14Uq34VMLTiilppuJsvIGV-9h9L.54uIVfw35eLz2hpaufDv3WDOLhGZTtyQRlgPCJXAZinbWMzFWi2z~EHEmCdTw4sVn0SLr5ZOd0g4IaSGlgJYQAw5hLxMul8F7Vyq-9h9L.31jN31000404040404040404040404040000000001jNjNjNjNjNjNjNjN00000000000001980000000000000008100000000auauauauauauauauauauauauauauau-9h9L.31jN3103040404040404040404040404010003333030o3V1no30o3V1i301i331i30003000303VfrVo0v9Xo00000or50bvauauauauauauauauauauauauauauau-_ddi96VR873bFVDkYqgnXn.2l2gGFo6IUYrwIlmuG3fM5~4Ko8E_jwkcRWSL2Kel5BVilGLd_4ZSFicMGdWgSK3~HPn-exsMpuiqMUIdp.18TZ1_taQY1vCOJ0uXLnFiA5UZQEEmhA5w4b8VVZ8uLoB3emG88p3B89Tf8xbziYgK71T.01Fk7CvETRJqiN9zsblnEziuyryl96JlTXmGJ4Qaqoc2Q6o2ryvvgwz